In [ ]:
# 读取文件夹结构
import os
# 指定路径
root_path = r"/Users/zhangnan/日常文件/ai/每周下载和预测/2 20250908/20250908-mslp/20250908-mslp-1/mslp周"
# 遍历文件夹结构
for root, dirs, files in os.walk(root_path):    
    print(f"当前文件夹: {root}")        
    # 判断是否有文件    
    if files:        
        display_files = files[:7]        
        print(f"  ✅ 文件数量: {len(files)}，展示前七个文件:")        
        for file in display_files:            
            print(f"    📄 {file}")    
        else:        
            print("  ⚠️ 该文件夹中没有文件")        
            if dirs:            
                print(f"  📁 包含的子文件夹: {', '.join(dirs)}")        
            else:            
                print("  ❌ 也没有子文件夹")    
        print()  # 空行用于分隔输出


In [ ]:
# 读取nc文件信息
import xarray as xr
# 定义文件路径
file_path = '/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-tas/20260831-tas-1/预测结果-2025年09月.nc'
# 打印原文件路径
print(f"文件路径: {file_path}")
# 打开NetCDF文件
ds = xr.open_dataset(file_path)
# 输出数据集的基本信息
print(ds)
# 如果需要查看数据集的变量列表，可以使用
#print(ds.variables)
# 如果需要查看数据集的维度，可以使用
#print(ds.dims)


In [ ]:
# 1 处理月最新数据
import xarray as xr
import numpy as np
import os

# 输入和输出路径
input_path = "/Users/zhangnan/日常文件/ai/每周下载和预测/2 20250908/20250908-mslp/20250908-mslp 数据/ERA5-monthly-single level-AZN-20250908.nc"
output_path = "/Users/zhangnan/日常文件/ai/每周下载和预测/2 20250908/20250908-mslp/20250908-mslp-1/merged_1.5deg_no_norm_20250908.nc"

# 自动创建输出目录
os.makedirs(os.path.dirname(output_path), exist_ok=True)

# 打开数据集
ds = xr.open_dataset(input_path)

# 要处理的变量列表（根据你的数据变量）
variables = ["u10", "v10", "t2m", "msl", "sst"]

all_vars = {}
ref_grid = None
ref_time = None

for varname in variables:
    print(f"📥 正在处理变量: {varname}")
    var_data = ds[varname]

    # 降采样，步长6，约1.5度分辨率
    sub = var_data.isel(latitude=slice(None, None, 6), longitude=slice(None, None, 6))

    # 保存参考网格和时间
    if ref_grid is None:
        ref_grid = sub
        ref_time = sub.valid_time if "valid_time" in sub.coords else sub.time if "time" in sub.coords else None
    else:
        # 对齐空间和时间坐标
        sub = sub.interp_like(ref_grid)

    # 用全局平均填充 NaN
    values = sub.values
    mask = np.isnan(values)
    if np.any(mask):
        mean_val = np.nanmean(values)
        values[mask] = mean_val
        sub.values = values

    all_vars[varname] = sub

# 合并变量为Dataset
merged_ds = xr.Dataset(all_vars)

# 重命名时间维度为 time（如果是valid_time）
if "valid_time" in merged_ds.dims or "valid_time" in merged_ds.coords:
    merged_ds = merged_ds.rename({"valid_time": "time"})

# 清理重复时间索引
merged_ds = merged_ds.sortby("time")
merged_ds = merged_ds.sel(time=~merged_ds.get_index("time").duplicated())

# 保存
merged_ds.to_netcdf(output_path)
print(f"✅ 保存完毕: {output_path}")


In [ ]:
# 2 推理2025年9月的数据
import os
import torch
import xarray as xr
import numpy as np
from tqdm import tqdm
import torch.nn as nn

# === 路径与参数 ===
DATA_PATH = "/Users/zhangnan/日常文件/ai/每周下载和预测/2 20250908/20250908-mslp/20250908-mslp-1/merged_1.5deg_no_norm_20250908.nc"
MODEL_PATH = "/mnt/g/次季节模型/提交准备/3 预测最新数据/climate_tcn_with_timeencoding_150_2month.pth"
OUTPUT_NC = "/Users/zhangnan/日常文件/ai/每周下载和预测/2 20250908/20250908-mslp/20250908-mslp-1/预测结果-2025年09月.nc"
timesteps = 15
variables = ["u10", "v10", "sst", "t2m", "msl"]
target_names = ["t2m_pred", "msl_pred"]
device = torch.device("cpu")

# === 加载数据 ===
ds = xr.open_dataset(DATA_PATH)
time = ds.time.values
H, W = ds.dims["latitude"], ds.dims["longitude"]

# === 选择 2024-05 到 2025-07 的 15 个月 ===
sel_ds = ds.sel(time=slice("2024-05-01", "2025-07-01"))
assert sel_ds.dims["time"] == timesteps, f"时间维度应为15，实际为 {sel_ds.dims['time']}"

data_stack = np.stack([sel_ds[var].values for var in variables], axis=1)  # (15, 5, H, W)

# === 时间编码（sin/cos 月份）===
month = (sel_ds.time.dt.month.values - 1).astype(np.float32)
month_sin = np.sin(2 * np.pi * month / 12)
month_cos = np.cos(2 * np.pi * month / 12)
month_sin_2d = np.tile(month_sin[:, None, None, None], (1, 1, H, W))
month_cos_2d = np.tile(month_cos[:, None, None, None], (1, 1, H, W))
data_stack = np.concatenate([data_stack, month_sin_2d, month_cos_2d], axis=1)  # (15, 7, H, W)

# === 标准化前5个变量 ===
mean = data_stack[:, :5].mean(axis=(0, 2, 3), keepdims=True)
std = data_stack[:, :5].std(axis=(0, 2, 3), keepdims=True)
std[std == 0] = 1.0
data_stack[:, :5] = (data_stack[:, :5] - mean) / std

# === 扩展维度 ===
x = torch.tensor(data_stack[None, ...], dtype=torch.float32).to(device)  # (1, 15, 7, H, W)

# === 模型结构（与训练一致） ===
class ChEncoder(nn.Module):
    def __init__(self, in_channels, embed_dim):
        super().__init__()
        self.spatial_encoder = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, padding=1), nn.ReLU(),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4))
        )
        self.fc = nn.Linear(32 * 4 * 4, embed_dim)

    def forward(self, x):
        x = self.spatial_encoder(x)
        return self.fc(x.flatten(1))

class TemporalConvNet(nn.Module):
    def __init__(self, input_size, num_channels, kernel_size=2, dropout=0.2):
        super().__init__()
        layers = []
        for i in range(len(num_channels)):
            dilation = 2 ** i
            in_ch = input_size if i == 0 else num_channels[i - 1]
            out_ch = num_channels[i]
            layers += [
                nn.Conv1d(in_ch, out_ch, kernel_size, padding=dilation, dilation=dilation),
                nn.ReLU(), nn.Dropout(dropout)
            ]
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

class ClimateTCNModel(nn.Module):
    def __init__(self, input_channels, height, width, timesteps, embed_dim=64):
        super().__init__()
        self.encoder = ChEncoder(input_channels, embed_dim)
        self.tcn = TemporalConvNet(embed_dim, [128, 128])
        self.fc = nn.Sequential(
            nn.Linear(128, 256), nn.ReLU(),
            nn.Linear(256, 2 * 2 * height * width)
        )
        self.height = height
        self.width = width

    def forward(self, x):  # (B, T, C, H, W)
        B, T, C, H, W = x.shape
        x = x.view(B * T, C, H, W)
        x = self.encoder(x)
        x = x.view(B, T, -1).transpose(1, 2)
        x = self.tcn(x)
        x = x[:, :, -1]
        out = self.fc(x)
        return out.view(B, 2, 2, H, W)  # (B, time, var, H, W)

# === 加载模型 ===
model = ClimateTCNModel(input_channels=7, height=H, width=W, timesteps=timesteps).to(device)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()

# === 推理 ===
with torch.no_grad():
    pred = model(x).cpu().numpy()[0]  # (2, 2, H, W)
"""
# === 反标准化（预测结果）===
t2m_mean, t2m_std = mean[0, 3, 0, 0], std[0, 3, 0, 0]
msl_mean, msl_std = mean[0, 4, 0, 0], std[0, 4, 0, 0]
pred[1, 0] = pred[1, 0] * t2m_std + t2m_mean  # t2m
pred[1, 1] = pred[1, 1] * msl_std + msl_mean  # msl
"""
# === 保存第2个月（即 2025-09） ===
pred_ds = xr.Dataset(
    {
        target_names[0]: (("time", "latitude", "longitude"), pred[1, 0][None, ...]),
        target_names[1]: (("time", "latitude", "longitude"), pred[1, 1][None, ...]),
    },
    coords={
        "time": [np.datetime64("2025-09-01")],
        "latitude": ds.latitude.values,
        "longitude": ds.longitude.values,
    }
)

os.makedirs(os.path.dirname(OUTPUT_NC), exist_ok=True)
pred_ds.to_netcdf(OUTPUT_NC)
print(f"✅ 已保存预测结果至：{OUTPUT_NC}")


In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# === 加载数据 ===
file_path = "/Users/zhangnan/日常文件/ai/每周下载和预测/2 20250908/20250908-mslp/20250908-mslp-1/预测结果-2025年09月.nc"
ds = xr.open_dataset(file_path)

t2m = ds['t2m_pred'].isel(time=0)  # (lat, lon)
msl = ds['msl_pred'].isel(time=0)
lat = ds.latitude.values
lon = ds.longitude.values

# === 创建投影图（两张子图） ===
fig = plt.figure(figsize=(16, 6))

# -- T2M --
ax1 = fig.add_subplot(1, 2, 1, projection=ccrs.PlateCarree())
t2m_plot = ax1.pcolormesh(lon, lat, t2m, cmap='coolwarm', shading='auto')
ax1.coastlines()
ax1.set_title("Predicted 2m Temperature (T2M) - July 2025", fontsize=14)
plt.colorbar(t2m_plot, ax=ax1, orientation='horizontal', pad=0.05, label="T2M (normalized)")

# -- MSLP --
ax2 = fig.add_subplot(1, 2, 2, projection=ccrs.PlateCarree())
msl_plot = ax2.pcolormesh(lon, lat, msl, cmap='viridis', shading='auto')
ax2.coastlines()
ax2.set_title("Predicted Mean Sea Level Pressure (MSLP) - July 2025", fontsize=14)
plt.colorbar(msl_plot, ax=ax2, orientation='horizontal', pad=0.05, label="MSLP (normalized)")

plt.tight_layout()
plt.show()


In [ ]:
# 3 重采样MSLP era5-daily到周的1.5度
import os
import xarray as xr
import numpy as np

# 输入文件路径
input_file = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/数据/data/ERA5-daily-single level-MSLP-20260831.nc"

# 输出目录
output_base_dir = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-mslp/20260831-mslp-1/mslp周"

# 目标插值的经纬度
target_lat = np.linspace(90, -90, 121)
target_lon = np.linspace(0, 358.5, 240)

# 打开数据集
ds = xr.open_dataset(input_file)
msl = ds["msl"]  # (valid_time, latitude, longitude)

# 保持单位 Pa
msl_pa = msl
msl_pa.name = "mslp"
msl_pa.attrs["units"] = "Pa"
msl_pa.attrs["long_name"] = "Mean sea level pressure"
msl_pa.attrs["standard_name"] = "air_pressure_at_mean_sea_level"

# 每周重采样，以周一为起始
msl_weekly = msl_pa.resample(valid_time="1W-MON").mean()

# 插值到目标网格
msl_interp = msl_weekly.interp(latitude=target_lat, longitude=target_lon, method="linear")

# 遍历每一周，保存为独立文件
for i in range(msl_interp.valid_time.size):
    da = msl_interp.isel(valid_time=i)
    date_str = np.datetime_as_string(da.valid_time.values, unit="D").replace("-", "")

    # 输出路径
    output_path = os.path.join(output_base_dir, f"obs-era5-{date_str}-mslp.nc")
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    # 构造 Dataset
    da_ds = xr.Dataset({da.name: da})
    da_ds = da_ds.expand_dims("time")
    da_ds = da_ds.assign_coords(time=[da.valid_time.values])
    da_ds["time"].attrs = {
        "standard_name": "time",
        "long_name": "time",
        "bounds": "time_bnds",
        "axis": "T"
    }

    da_ds["variable"] = xr.DataArray(["mslp"], dims="variable")
    da_ds["variable"].attrs = {
        "long_name": "variable name",
        "standard_name": "variable",
    }

    # 保存为 NetCDF 文件
    da_ds.to_netcdf(output_path)

print(f"✅ 处理完成，周数据已保存到 {output_base_dir}")


In [ ]:
# 4 处理到周的900hPa-Divergence
import os
import xarray as xr
import numpy as np

# 输入文件路径
input_file = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/数据/data/ERA5-daily-900hPa-Divergence-20260831.nc"
# 输出文件路径
output_path = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-mslp/20260831-mslp-1/obs-era5-20260831-div900.nc"

# 目标插值网格
target_lat = np.linspace(90, -90, 121)
target_lon = np.linspace(0, 358.5, 240)

# 打开数据集
ds = xr.open_dataset(input_file)
div = ds['d'].sel(pressure_level=900)  # 取 900hPa 层

# 设置变量属性
div.name = 'divergence900'
div.attrs['units'] = ds['d'].attrs.get('units', 's^-1')
div.attrs['long_name'] = 'Divergence at 900hPa'
div.attrs['standard_name'] = 'divergence'

# 按周重采样（周一开始）
div_weekly = div.resample(valid_time='1W-MON').mean()

# 插值到目标网格
div_interp = div_weekly.interp(latitude=target_lat, longitude=target_lon, method='linear')

# 转换为 Dataset 并设置时间维度
div_ds = div_interp.to_dataset(name='divergence900')
div_ds = div_ds.rename({'valid_time': 'time'})  # 改为标准时间维度名称

# 添加 time 属性
div_ds['time'].attrs = {
    'standard_name': 'time',
    'long_name': 'time',
    'axis': 'T'
}

# 添加 variable 坐标
div_ds['variable'] = xr.DataArray(['divergence900'], dims='variable')
div_ds['variable'].attrs = {
    'long_name': 'variable name',
    'standard_name': 'variable',
}

# 保存合并后的文件
os.makedirs(os.path.dirname(output_path), exist_ok=True)
div_ds.to_netcdf(output_path)

print("✅ 处理完成！文件已保存到：", output_path)


In [ ]:
# 5 处理到周的900hPa-PotentialVorticity
import os
import xarray as xr
import numpy as np

# 输入文件路径
input_file = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/数据/data/ERA5-daily-900hPa-PotentialVorticity-20260831.nc"

# 输出文件路径
output_file = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-mslp/20260831-mslp-1/obs-era5-20260831-pv900.nc"

# 目标插值的经纬度
target_lat = np.linspace(90, -90, 121)
target_lon = np.linspace(0, 358.5, 240)

# 打开数据集并提取 900hPa 位势涡度
ds = xr.open_dataset(input_file)
pv = ds['pv'].sel(pressure_level=900)  # (valid_time, latitude, longitude)

# 设置变量属性
pv.name = 'pv900'
pv.attrs['units'] = ds['pv'].attrs.get('units', 'K m^2 kg^-1 s^-1')
pv.attrs['long_name'] = 'Potential Vorticity at 900hPa'
pv.attrs['standard_name'] = 'potential_vorticity'

# 按周重采样（以周一为起始），取均值
pv_weekly = pv.resample(valid_time='1W-MON').mean()

# 插值到目标网格
pv_interp = pv_weekly.interp(latitude=target_lat, longitude=target_lon, method='linear')

# 构造数据集
da_ds = xr.Dataset({'pv900': pv_interp})

# 设置时间维度
da_ds = da_ds.rename({'valid_time': 'time'})
da_ds['time'].attrs = {
    'standard_name': 'time',
    'long_name': 'time',
    'bounds': 'time_bnds',
    'axis': 'T'
}

# 添加 variable 坐标
da_ds['variable'] = xr.DataArray(['pv900'], dims='variable')
da_ds['variable'].attrs = {
    'long_name': 'variable name',
    'standard_name': 'variable',
}

# 创建输出目录
os.makedirs(os.path.dirname(output_file), exist_ok=True)

# 保存为单一 NetCDF 文件
da_ds.to_netcdf(output_file)

print(f"合并后的文件已保存至: {output_file}")


In [ ]:
# 6 处理到周的700hPa-SpecificHumidity
import os
import xarray as xr
import numpy as np

# 输入文件
input_file = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/数据/data/ERA5-daily-700hPa-SpecificHumidity-20260831.nc"

# 输出文件
output_file = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-mslp/20260831-mslp-1/obs-era5-20260831-q700.nc"

# 目标插值网格
target_lat = np.linspace(90, -90, 121)
target_lon = np.linspace(0, 358.5, 240)

# 打开数据集
ds = xr.open_dataset(input_file)

# 选择 700hPa 比湿
q700 = ds['q'].sel(pressure_level=700).squeeze()
q700.name = 'q700'
q700.attrs['units'] = 'kg kg-1'
q700.attrs['long_name'] = 'Specific humidity at 700hPa'
q700.attrs['standard_name'] = 'specific_humidity'

# 按周平均 (以周一为起点)
q700_weekly = q700.resample(valid_time='1W-MON').mean()

# 插值到目标网格
q700_interp = q700_weekly.interp(latitude=target_lat, longitude=target_lon, method='linear')

# 创建 Dataset
ds_out = xr.Dataset({'q700': q700_interp})

# 处理时间坐标
ds_out = ds_out.rename({'valid_time': 'time'})
ds_out['time'].attrs = {
    'standard_name': 'time',
    'long_name': 'time',
    'bounds': 'time_bnds',
    'axis': 'T'
}

# 添加 variable 坐标
ds_out['variable'] = xr.DataArray(['q700'], dims='variable')
ds_out['variable'].attrs = {
    'long_name': 'variable name',
    'standard_name': 'variable'
}

# 确保输出目录存在
os.makedirs(os.path.dirname(output_file), exist_ok=True)

# 保存到 NetCDF
ds_out.to_netcdf(output_file)

print(f"已保存到 {output_file}")


In [ ]:
# 7 850-500hpa
import os
import xarray as xr
import numpy as np
import pandas as pd

# 输入文件路径
file_850 = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/数据/data/ERA5-daily-850hPa-Geopotential-20260831.nc"
file_500 = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/数据/data/ERA5-daily-500hPa-Geopotential-20260831.nc"

# 输出文件路径
output_file = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-mslp/20260831-mslp-1/obs-era5-20260831-gh.nc"

# 目标插值网格
target_lat = np.arange(90, -91, -1.5)
target_lon = np.arange(0, 360, 1.5)

# 读取数据
ds850 = xr.open_dataset(file_850)
ds500 = xr.open_dataset(file_500)

# 转为位势高度，去掉 pressure_level
z850 = ds850['z'].squeeze() / 9.8
z500 = ds500['z'].squeeze() / 9.8

# 插值到目标网格
z850_interp = z850.interp(latitude=target_lat, longitude=target_lon, method="linear")
z500_interp = z500.interp(latitude=target_lat, longitude=target_lon, method="linear")

# 缺失值填充
z850_filled = z850_interp.ffill('latitude').bfill('latitude').ffill('longitude').bfill('longitude')
z500_filled = z500_interp.ffill('latitude').bfill('latitude').ffill('longitude').bfill('longitude')

# 计算差值
z500_z850 = z500_filled - z850_filled

# 时间维度
time_dim = 'valid_time' if 'valid_time' in z850_filled.dims else 'time'

# 按周重采样 (每周一)
z850_weekly = z850_filled.resample({time_dim: '1W-MON'}).mean()
z500_z850_weekly = z500_z850.resample({time_dim: '1W-MON'}).mean()

# 合并成一个 Dataset
ds_out = xr.Dataset({
    "z850_weekly": z850_weekly,
    "z500_z850_weekly": z500_z850_weekly,
})

# 保存到单个 NetCDF 文件
os.makedirs(os.path.dirname(output_file), exist_ok=True)
ds_out.to_netcdf(output_file)

print(f"✅ 已保存合并文件: {output_file}")
print(f"包含 {len(ds_out[time_dim])} 周数据")
print(f"z850_weekly 范围: {ds_out['z850_weekly'].min().item():.3f} ~ {ds_out['z850_weekly'].max().item():.3f}")
print(f"z500_z850_weekly 范围: {ds_out['z500_z850_weekly'].min().item():.3f} ~ {ds_out['z500_z850_weekly'].max().item():.3f}")


In [ ]:
# 8 合并周数据
import os
import xarray as xr
from tqdm import tqdm

# 输入文件路径
base_dir = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-mslp/20260831-mslp-1/"
mslp_dir = os.path.join(base_dir, "mslp周")
gh_file = os.path.join(base_dir, "obs-era5-20260831-gh.nc")
q700_file = os.path.join(base_dir, "obs-era5-20260831-q700.nc")
pv900_file = os.path.join(base_dir, "obs-era5-20260831-pv900.nc")
div900_file = os.path.join(base_dir, "obs-era5-20260831-div900.nc")

# 输出文件
output_path = os.path.join(base_dir, "mslp-weekmerged_20260831.nc")

def drop_pressure_level(ds):
    """安全移除 pressure_level"""
    if "pressure_level" in ds.variables:
        ds = ds.drop_vars("pressure_level")
    elif "pressure_level" in ds.coords:
        ds = ds.reset_coords("pressure_level", drop=True)
    return ds

def merge_weekly_files():
    print("开始处理 20260831 周数据合并...")

    # 读取单变量的周文件
    mslp_files = sorted([f for f in os.listdir(mslp_dir) if f.endswith(".nc")])
    merged_datasets = []

    # 打开整段文件
    ds_gh = drop_pressure_level(xr.open_dataset(gh_file))
    ds_q700 = drop_pressure_level(xr.open_dataset(q700_file))
    ds_pv900 = drop_pressure_level(xr.open_dataset(pv900_file))
    ds_div900 = drop_pressure_level(xr.open_dataset(div900_file))

    # 遍历每个周文件并处理
    for i, mslp_file in enumerate(tqdm(mslp_files, desc="合并进度")):
        try:
            ds_mslp = drop_pressure_level(xr.open_dataset(os.path.join(mslp_dir, mslp_file)))

            # 时间
            time_val = ds_mslp['time'].values

            # 提取变量并扩展维度
            mslp_sel = ds_mslp['mslp'].isel(time=0).expand_dims({'time': time_val})
            z850_weekly = ds_gh['z850_weekly'].isel(valid_time=i).expand_dims({'time': time_val})
            z500_z850_weekly = ds_gh['z500_z850_weekly'].isel(valid_time=i).expand_dims({'time': time_val})
            q700 = ds_q700['q700'].isel(time=i).expand_dims({'time': time_val})
            divergence900 = ds_div900['divergence900'].isel(time=i).expand_dims({'time': time_val})
            pv900 = ds_pv900['pv900'].isel(time=i).expand_dims({'time': time_val})

            # 构建数据集
            ds_merged = xr.Dataset(
                data_vars={
                    'mslp': mslp_sel,
                    'z850_weekly': z850_weekly,
                    'z500_z850_weekly': z500_z850_weekly,
                    'q700': q700,
                    'divergence900': divergence900,
                    'pv900': pv900,
                },
                coords={
                    'time': time_val,
                    'latitude': ds_mslp['latitude'],
                    'longitude': ds_mslp['longitude'],
                }
            )
            merged_datasets.append(ds_merged)
            ds_mslp.close()

        except Exception as e:
            print(f"  ❌ Error on {mslp_file}: {e}")

    # 检查是否有数据
    if not merged_datasets:
        print("❌ 没有成功合并的数据集，请检查文件")
        return

    # 使用 coords='minimal' 合并
    ds_all = xr.concat(merged_datasets, dim='time', compat='override', coords='minimal').sortby('time')

    # 保存输出
    print(f"✅ 保存合并文件到: {output_path}")
    ds_all.to_netcdf(output_path)
    ds_all.close()
    ds_gh.close()
    ds_q700.close()
    ds_pv900.close()
    ds_div900.close()

    print("🎯 数据合并完成")

merge_weekly_files()


In [ ]:
# 9 周数据添加基准日期
import xarray as xr
import pandas as pd
import numpy as np
import os

# 输入输出路径
input_file = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-mslp/20260831-mslp-1/mslp-weekmerged_20260831.nc"
output_dir = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-mslp/20260831-mslp-1/"
output_file = os.path.join(output_dir, "20260831_weekly_sample1.nc")

# 加载数据集并确保时间唯一
ds = xr.open_dataset(input_file)
ds['time'] = pd.to_datetime(ds['time'].values)
_, unique_idx = np.unique(ds['time'], return_index=True)
ds = ds.isel(time=unique_idx)

# 设置目标周日期
target_date = pd.to_datetime("2026-08-31")

# 构造过去 20 周（第4周到第23周）和 10 周（第4周到第13周）的日期列表
past_20_weeks = [target_date - pd.Timedelta(weeks=i) for i in range(4, 24)]
past_10_weeks = [target_date - pd.Timedelta(weeks=i) for i in range(4, 14)]

# 选择数据，自动忽略不存在的时间
mslp_hist_all = ds['mslp'].sel(time=past_20_weeks)
z850_hist_all = ds['z850_weekly'].sel(time=past_10_weeks)
z500_z850_hist_all = ds['z500_z850_weekly'].sel(time=past_10_weeks)
q700_hist_all = ds['q700'].sel(time=past_10_weeks)
divergence900_hist_all = ds['divergence900'].sel(time=past_10_weeks)
pv900_hist_all = ds['pv900'].sel(time=past_10_weeks)

# 构建新的 Dataset
data_vars = {}
coords = {"time": [target_date], "latitude": ds.latitude, "longitude": ds.longitude}

# 历史20周 mslp
for i in range(mslp_hist_all.time.size):
    data_vars[f"mslp_hist_{i}"] = (("latitude", "longitude"), mslp_hist_all.isel(time=i).values)

# 历史10周 z850_weekly
for i in range(z850_hist_all.time.size):
    data_vars[f"z850_hist_{i}"] = (("latitude", "longitude"), z850_hist_all.isel(time=i).values)

# 历史10周 z500_z850_weekly
for i in range(z500_z850_hist_all.time.size):
    data_vars[f"z500_z850_hist_{i}"] = (("latitude", "longitude"), z500_z850_hist_all.isel(time=i).values)

# 历史10周 q700
for i in range(q700_hist_all.time.size):
    data_vars[f"q700_hist_{i}"] = (("latitude", "longitude"), q700_hist_all.isel(time=i).values)

# 历史10周 divergence900
for i in range(divergence900_hist_all.time.size):
    data_vars[f"divergence900_hist_{i}"] = (("latitude", "longitude"), divergence900_hist_all.isel(time=i).values)

# 历史10周 pv900
for i in range(pv900_hist_all.time.size):
    data_vars[f"pv900_hist_{i}"] = (("latitude", "longitude"), pv900_hist_all.isel(time=i).values)

ds_sample = xr.Dataset(data_vars=data_vars, coords=coords)

# 保存到 NetCDF
os.makedirs(output_dir, exist_ok=True)
ds_sample.to_netcdf(output_file)
print(f"✅ 数据整理完成并保存：{output_file}")


In [ ]:
# 10 添加2月预测数据
import os
import xarray as xr
import numpy as np

# 输入文件
weekly_file = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-mslp/20260831-mslp-1/20260831_weekly_sample1.nc"
pred_file = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-tas/20260831-tas-1/预测结果-2026年8月.nc"

# 输出目录
output_dir = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-mslp/20260831-mslp-1/merged_with_pred"
os.makedirs(output_dir, exist_ok=True)

# 打开数据集
ds_weekly = xr.open_dataset(weekly_file)
ds_pred = xr.open_dataset(pred_file)

# 调整经度为 [0, 360] 并排序
ds_pred = ds_pred.assign_coords(longitude=(ds_pred.longitude % 360)).sortby("longitude")

# 取预测变量
pred_msl = ds_pred["msl_pred"]
pred_t2m = ds_pred["t2m_pred"]

# 将周数据时间对应到当月
time_week = ds_weekly["time"].values[0]
month_start = np.datetime64(f"{time_week.astype('datetime64[M]')}")  # 月首

# 检查预测数据时间是否存在
if month_start not in pred_msl.time:
    print(f"❌ 缺失该月份预测数据: {month_start}")
    shape = (len(ds_weekly.latitude), len(ds_weekly.longitude))
    pred_msl_values = np.full(shape, np.nan)
    pred_t2m_values = np.full(shape, np.nan)
else:
    pred_msl_values = pred_msl.sel(time=month_start).values
    pred_t2m_values = pred_t2m.sel(time=month_start).values

# 构建 DataArray
pred_msl_da = xr.DataArray(
    data=pred_msl_values[np.newaxis, :, :],  # 增加时间维度
    dims=("time", "latitude", "longitude"),
    coords={
        "time": ds_weekly.time,
        "latitude": ds_weekly.latitude,
        "longitude": ds_weekly.longitude
    },
    name="pred_msl_month",
    attrs={"description": "对应周的月份预测海平面气压"}
)

pred_t2m_da = xr.DataArray(
    data=pred_t2m_values[np.newaxis, :, :],
    dims=("time", "latitude", "longitude"),
    coords={
        "time": ds_weekly.time,
        "latitude": ds_weekly.latitude,
        "longitude": ds_weekly.longitude
    },
    name="pred_t2m_month",
    attrs={"description": "对应周的月份预测2米温度"}
)

# 添加到周数据集
ds_weekly["pred_msl_month"] = pred_msl_da
ds_weekly["pred_t2m_month"] = pred_t2m_da

# 保存
output_file = os.path.join(output_dir, "mslp_weekly_with_monthly_pred.nc")
ds_weekly.to_netcdf(output_file)
print(f"💾 已保存: {output_file}")


In [ ]:
#提前1？？月预测添加月预测数据
import os
import xarray as xr
import numpy as np

# 输入文件
weekly_file = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-mslp/20260831-mslp-1/20260831_weekly_sample1.nc"
pred_file = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-tas/20260831-tas-1/预测结果-2025年12月.nc"

# 输出目录
output_dir = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-mslp/20260831-mslp-1/merged_with_pred"
os.makedirs(output_dir, exist_ok=True)

# 打开数据集
ds_weekly = xr.open_dataset(weekly_file)
ds_pred = xr.open_dataset(pred_file)

# 调整经度为 [0, 360] 并排序
ds_pred = ds_pred.assign_coords(longitude=(ds_pred.longitude % 360)).sortby("longitude")

# 将周数据时间对应到当月
time_week = ds_weekly["time"].values[0]
month_start = np.datetime64(f"{time_week.astype('datetime64[M]')}")  # 月首

# 检查预测数据时间是否存在
if month_start not in ds_pred.time.values:
    print(f"❌ 缺失该月份预测数据: {month_start}")
    shape = (len(ds_weekly.latitude), len(ds_weekly.longitude))
    pred_msl_values = np.full(shape, np.nan)
    pred_t2m_values = np.full(shape, np.nan)
else:
    # 直接获取预测数据值（没有时间维度）
    pred_msl_values = ds_pred["pred_msl"].values
    pred_t2m_values = ds_pred["pred_t2m"].values

# 构建 DataArray
pred_msl_da = xr.DataArray(
    data=pred_msl_values[np.newaxis, :, :],  # 增加时间维度
    dims=("time", "latitude", "longitude"),
    coords={
        "time": ds_weekly.time,
        "latitude": ds_weekly.latitude,
        "longitude": ds_weekly.longitude
    },
    name="pred_msl_month",
    attrs={"description": "对应周的月份预测海平面气压"}
)

pred_t2m_da = xr.DataArray(
    data=pred_t2m_values[np.newaxis, :, :],
    dims=("time", "latitude", "longitude"),
    coords={
        "time": ds_weekly.time,
        "latitude": ds_weekly.latitude,
        "longitude": ds_weekly.longitude
    },
    name="pred_t2m_month",
    attrs={"description": "对应周的月份预测2米温度"}
)

# 添加到周数据集
ds_weekly["pred_msl_month"] = pred_msl_da
ds_weekly["pred_t2m_month"] = pred_t2m_da

# 保存
output_file = os.path.join(output_dir, "mslp_weekly_with_monthly_pred.nc")
ds_weekly.to_netcdf(output_file)
print(f"💾 已保存: {output_file}")

# 关闭数据集
ds_weekly.close()
ds_pred.close()

In [ ]:
# 11 最后一步，添加地形数据
import xarray as xr
import numpy as np
import os

# 文件路径
input_file = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-mslp/20260831-mslp-1/merged_with_pred/mslp_weekly_with_monthly_pred.nc"
output_file = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-mslp/20260831-mslp-1/merged_with_pred/mslp_weekly_with_monthly_pred_with_elev.nc"

# 地形数据
elev_file = "/Users/zhangnan/日常文件/ai/每周下载和预测/推理用/地形数据/etopo_1.5deg.nc"
ds_elev = xr.open_dataset(elev_file)

# 将地形经度转换为 0~360 范围
def lon_180_to_360(lon):
    lon_360 = lon.copy()
    lon_360 = np.where(lon_360 < 0, lon_360 + 360, lon_360)
    return lon_360

ds_elev = ds_elev.assign_coords(lon=lon_180_to_360(ds_elev.lon))
ds_elev = ds_elev.sortby('lon')

# 打开目标文件
ds = xr.open_dataset(input_file)

# 重命名地形坐标与目标一致
ds_elev_renamed = ds_elev.rename({'lat': 'latitude', 'lon': 'longitude'})

# 插值到目标经纬度
ds_elev_interp = ds_elev_renamed.interp(
    latitude=ds.latitude,
    longitude=ds.longitude,
    method="linear"
)

# 扩展到时间维度
elev_expanded = ds_elev_interp['elevation'].expand_dims({'time': ds.time}, axis=0)
elev_expanded = elev_expanded.transpose('time', 'latitude', 'longitude')

# 新建 DataArray
elev_da = xr.DataArray(
    data=elev_expanded.values,
    dims=['time', 'latitude', 'longitude'],
    coords={'time': ds.time, 'latitude': ds.latitude, 'longitude': ds.longitude},
    attrs={
        'long_name': 'surface_elevation',
        'units': 'meters',
        'description': 'Surface elevation interpolated from etopo_1.5deg.nc and expanded along time dimension'
    }
)

# 添加到数据集
ds = ds.assign(elevation=elev_da)

# 保存新文件
ds.to_netcdf(output_file)
ds.close()

print(f"完成！新文件已保存: {output_file}")


In [ ]:
# 读取nc文件信息
import xarray as xr
# 定义文件路径
file_path = '/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-mslp/20260831-mslp-1/merged_with_pred/mslp_weekly_with_monthly_pred_with_elev.nc'
# 打印原文件路径
print(f"文件路径: {file_path}")
# 打开NetCDF文件
ds = xr.open_dataset(file_path)
# 输出数据集的基本信息
print(ds)
# 如果需要查看数据集的变量列表，可以使用
print(ds.variables)
# 如果需要查看数据集的维度，可以使用
print(ds.dims)


In [ ]:
# 读取nc文件信息
import xarray as xr
# 定义文件路径
file_path = '/Users/zhangnan/日常文件/ai/每周下载和预测/14 20251201/20251201-tas/20251201-tas-1/预测结果-2025年12月.nc'
# 打印原文件路径
print(f"文件路径: {file_path}")
# 打开NetCDF文件
ds = xr.open_dataset(file_path)
# 输出数据集的基本信息
print(ds)
# 如果需要查看数据集的变量列表，可以使用
#print(ds.variables)
# 如果需要查看数据集的维度，可以使用
#print(ds.dims)
